<a href="https://colab.research.google.com/github/chiyanglin-AStar/Python_3D_Lib/blob/main/Numba_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 在 Google Colab 中使用 Numba 教學

Numba 是一個開源的 JIT (Just-In-Time) 編譯器，它可以將 Python 程式碼編譯成快速的機器碼。這對於處理大量數值計算的程式碼，例如科學運算、數據分析等，能顯著提升執行效率。Numba 特別適用於迴圈和數學運算，因為它可以將這些部分優化，使其接近 C 或 Fortran 的執行速度。

## 1. 安裝 Numba

在 Google Colab 中安裝 Numba 非常簡單，只需要使用 `pip` 命令即可。

In [5]:
pip install numba

安裝完成後，您就可以在 Colab 中使用 Numba 了。通常會建議在安裝完畢後，重新啟動運行時（Runtime -> Restart runtime），以確保所有環境變數都已正確加載，儘管對於 Numba 來說，通常不是必需的步驟，但這是一個好的習慣。

## 2. Numba 的基本使用與特性示範

Numba 最核心的特性是其 `@jit` 裝飾器 (decorator)，它可以將一個 Python 函數即時編譯成機器碼。我們將透過一個簡單的例子來示範 Numba 如何提升效能。

### 範例一：計算數組元素的平方和

我們將定義一個函數來計算一個大型數組中所有元素的平方和，並比較使用 Numba 和不使用 Numba 的效能差異。

In [6]:
import numpy as np
import time

# 產生一個大型的 NumPy 數組
size = 10**7
arr = np.arange(size, dtype=np.float64)

# --- 不使用 Numba 的函數 ---
def sum_square_pure_python(a):
    total = 0.0
    for x in a:
        total += x * x
    return total

print("正在執行不使用 Numba 的函數...")
start_time = time.time()
result_pure = sum_square_pure_python(arr)
end_time = time.time()
print(f"不使用 Numba 的結果: {result_pure}")
print(f"不使用 Numba 的執行時間: {end_time - start_time:.4f} 秒")

正在執行不使用 Numba 的函數...
不使用 Numba 的結果: 3.333332833337171e+20
不使用 Numba 的執行時間: 2.3267 秒


#### 程式說明：

-   `import numpy as np`: 導入 NumPy 庫用於數組操作。
-   `import time`: 導入 `time` 庫用於測量函數執行時間。
-   `size = 10**7`: 定義數組大小為一千萬。
-   `arr = np.arange(size, dtype=np.float64)`: 創建一個包含一千萬個浮點數的 NumPy 數組。
-   `sum_square_pure_python(a)`: 這是一個普通的 Python 函數，它遍歷數組 `a` 中的每個元素，計算其平方並累加到 `total` 中。此函數沒有任何 Numba 優化。

In [7]:
from numba import jit

# --- 使用 Numba 的函數 ---
@jit(nopython=True) # 使用 @jit 裝飾器啟用 Numba 優化
def sum_square_numba(a):
    total = 0.0
    for x in a:
        total += x * x
    return total

print("\n正在執行使用 Numba 的函數 (首次調用會包含編譯時間)...")
start_time = time.time()
result_numba = sum_square_numba(arr) # 第一次調用會觸發 JIT 編譯
end_time = time.time()
print(f"使用 Numba 的結果: {result_numba}")
print(f"使用 Numba 的執行時間 (首次編譯): {end_time - start_time:.4f} 秒")

print("\n再次執行使用 Numba 的函數 (已編譯，顯示真實運行時間)...")
start_time = time.time()
result_numba = sum_square_numba(arr) # 第二次調用直接使用編譯好的機器碼
end_time = time.time()
print(f"使用 Numba 的結果: {result_numba}")
print(f"使用 Numba 的執行時間 (編譯後): {end_time - start_time:.4f} 秒")


正在執行使用 Numba 的函數 (首次調用會包含編譯時間)...
使用 Numba 的結果: 3.333332833337171e+20
使用 Numba 的執行時間 (首次編譯): 0.1149 秒

再次執行使用 Numba 的函數 (已編譯，顯示真實運行時間)...
使用 Numba 的結果: 3.333332833337171e+20
使用 Numba 的執行時間 (編譯後): 0.0194 秒


### 範例二：使用 `vectorize` 裝飾器加速元素級運算

Numba 的 `vectorize` 裝飾器允許您將一個標量（單一值）函數轉換為一個 NumPy universal function (ufunc)。這對於對 NumPy 數組進行元素級運算非常有用，並且可以避免 Python 的迴圈開銷。

In [8]:
from numba import vectorize

# 定義一個標量函數
def my_func(x, y):
    return x * x + y * y + 1

# 使用 Numba 的 vectorize 裝飾器將其轉換為 ufunc
# 'float64(float64, float64)' 指定了輸入和輸出類型
@vectorize(['float64(float64, float64)']) # 函數簽名
def vectorized_my_func(x, y):
    return x * x + y * y + 1

# 產生大型 NumPy 數組
size_vec = 10**7
arr_x = np.arange(size_vec, dtype=np.float64)
arr_y = np.arange(size_vec, dtype=np.float64) * 2

print("正在執行不使用 Numba 的純 Python 元素級運算...")
start_time = time.time()
# 對大型數組進行元素級運算，這裡使用列表推導模擬純 Python 迴圈
result_pure_vec = [my_func(arr_x[i], arr_y[i]) for i in range(size_vec)]
end_time = time.time()
print(f"不使用 Numba 的執行時間: {end_time - start_time:.4f} 秒")

print("\n正在執行使用 Numba `vectorize` 的元素級運算...")
start_time = time.time()
result_numba_vec = vectorized_my_func(arr_x, arr_y)
end_time = time.time()
print(f"使用 Numba `vectorize` 的執行時間: {end_time - start_time:.4f} 秒")

# 確認結果是否一致 (可能會因為浮點數精度略有差異)
print(f"結果是否近似一致: {np.allclose(result_pure_vec, result_numba_vec)}")

正在執行不使用 Numba 的純 Python 元素級運算...
不使用 Numba 的執行時間: 7.9811 秒

正在執行使用 Numba `vectorize` 的元素級運算...
使用 Numba `vectorize` 的執行時間: 0.0493 秒
結果是否近似一致: True


#### 程式說明：

-   `@vectorize(['float64(float64, float64)'])`: 這是 `vectorize` 裝飾器的使用方式。
    -   `'float64(float64, float64)'` 是函數的簽名 (signature)。它定義了輸入參數和回傳值的資料型別。在這個例子中，它表示函數接受兩個 `float64` 型別的參數，並回傳一個 `float64` 型別的值。明確指定型別有助於 Numba 進行更高效的編譯。
-   `vectorized_my_func(x, y)`: 這裡定義的函數是一個標量函數，它會應用於每個單一的數值。
-   **效能比較**: 與 `jit` 類似，`vectorize` 會將標量函數編譯為高效的機器碼，並能直接應用於整個 NumPy 數組，避免了顯式的 Python 迴圈，從而大幅提升執行速度。您會注意到 `vectorized_my_func` 直接接受 NumPy 數組作為輸入，並回傳一個新的 NumPy 數組，其操作方式與 NumPy 內建的 ufuncs 相似。

### 範例三：使用 `prange` 進行平行運算

Numba 的 `prange` (parallel range) 是一種特殊的迴圈函數，它類似於 Python 內建的 `range`，但會指示 Numba 將迴圈的迭代平行化，以利用多核心 CPU 的優勢。要啟用 `prange`，您需要將 `@jit` 裝飾器的 `parallel` 參數設置為 `True`。

In [9]:
from numba import jit, prange
import numpy as np
import time

# 確保我們有 arr 變數，如果前面單元格未執行，可以重新創建
if 'arr' not in locals():
    size = 10**7
    arr = np.arange(size, dtype=np.float64)

# --- 使用 Numba jit 但不平行 (用於比較) ---
@jit(nopython=True)
def sum_square_numba_serial(a):
    total = 0.0
    for i in range(len(a)):
        total += a[i] * a[i]
    return total

# --- 使用 Numba jit 且平行 (prange) ---
@jit(nopython=True, parallel=True)
def sum_square_numba_parallel(a):
    total = 0.0
    # 使用 prange 而不是 range 來啟用平行迴圈
    for i in prange(len(a)):
        total += a[i] * a[i]
    return total

print("\n正在執行 Numba (非平行) 函數...")
start_time = time.time()
result_serial = sum_square_numba_serial(arr)
end_time = time.time()
print(f"Numba (非平行) 結果: {result_serial}")
print(f"Numba (非平行) 執行時間: {end_time - start_time:.4f} 秒")

print("\n正在執行 Numba (平行 prange) 函數 (首次調用會包含編譯時間)...")
start_time = time.time()
result_parallel = sum_square_numba_parallel(arr) # 第一次調用會觸發 JIT 編譯
end_time = time.time()
print(f"Numba (平行 prange) 結果: {result_parallel}")
print(f"Numba (平行 prange) 執行時間 (首次編譯): {end_time - start_time:.4f} 秒")

print("\n再次執行 Numba (平行 prange) 函數 (已編譯，顯示真實運行時間)...")
start_time = time.time()
result_parallel = sum_square_numba_parallel(arr) # 第二次調用直接使用編譯好的機器碼
end_time = time.time()
print(f"Numba (平行 prange) 結果: {result_parallel}")
print(f"Numba (平行 prange) 執行時間 (編譯後): {end_time - start_time:.4f} 秒")

# 確認結果是否一致
print(f"\n結果是否一致 (非平行 vs 平行): {np.allclose(result_serial, result_parallel)}")


正在執行 Numba (非平行) 函數...
Numba (非平行) 結果: 3.333332833337171e+20
Numba (非平行) 執行時間: 0.1970 秒

正在執行 Numba (平行 prange) 函數 (首次調用會包含編譯時間)...
Numba (平行 prange) 結果: 3.333332833336475e+20
Numba (平行 prange) 執行時間 (首次編譯): 0.5824 秒

再次執行 Numba (平行 prange) 函數 (已編譯，顯示真實運行時間)...
Numba (平行 prange) 結果: 3.333332833336475e+20
Numba (平行 prange) 執行時間 (編譯後): 0.0092 秒

結果是否一致 (非平行 vs 平行): True


#### 程式說明：

-   `from numba import prange`: 從 Numba 庫中導入 `prange` 函數。
-   `@jit(nopython=True, parallel=True)`: 這個裝飾器是啟用平行化處理的關鍵。
    -   `nopython=True`: 和之前的範例一樣，用於強制編譯為純機器碼以獲得最佳效能。
    -   `parallel=True`: 這個參數告訴 Numba 嘗試將函數內的迴圈平行化。
-   `for i in prange(len(a))`: 將標準的 `range` 替換為 `prange`，這就是指示 Numba 將此迴圈分解並在多個 CPU 核心上並行執行的指令。
-   **效能比較**：在支援多核心處理的環境中（例如 Colab），您會觀察到使用 `prange` 的函數在編譯後的執行時間通常會比單純使用 `@jit` 但未指定 `parallel=True` 的函數更快，尤其是在處理大型數據集時。這是因為計算任務被分散到多個核心上同時進行。

## 3. 使用 Numba CUDA 進行 GPU 加速

Numba 最強大的功能之一是它能夠將 Python 程式碼編譯成在 NVIDIA GPU 上執行的 CUDA 核心。這對於需要大量平行計算的任務（例如矩陣運算、深度學習前處理等）來說，能帶來巨大的效能提升。

在 Google Colab 中，您需要確保您的執行階段（Runtime）設定為 GPU。您可以透過 `Runtime` -> `Change runtime type` -> `Hardware accelerator` 選擇 `GPU`。

### 範例一：向量加法 (Vector Addition)

In [1]:
from numba import cuda
import numpy as np
import time

# 檢查是否有可用的 CUDA GPU
if not cuda.is_available():
    print("Numba 無法找到 CUDA GPU。請確保您的 Colab 執行階段已啟用 GPU (Runtime -> Change runtime type -> Hardware accelerator -> GPU)。")
else:
    print("Numba 已找到 CUDA GPU。")

# 定義 CUDA 核心 (Kernel Function)
# 這個函數會在每個 GPU 執行緒上執行
@cuda.jit
def vector_add_gpu(x, y, out):
    idx = cuda.grid(1) # 獲取當前執行緒的全局索引 (1D)
    if idx < len(x):
        out[idx] = x[idx] + y[idx]

# 資料大小
size_cuda = 10**7
arr_a = np.arange(size_cuda, dtype=np.float32)
arr_b = np.arange(size_cuda, dtype=np.float32) * 2

# --- 純 NumPy (CPU) 執行 ---
print("\n正在執行純 NumPy (CPU) 向量加法...")
start_time_cpu = time.time()
result_cpu = arr_a + arr_b
end_time_cpu = time.time()
print(f"純 NumPy 執行時間: {end_time_cpu - start_time_cpu:.4f} 秒")

# --- Numba CUDA (GPU) 執行 ---
print("\n正在執行 Numba CUDA (GPU) 向量加法...")

# 將資料從主機 (CPU) 複製到裝置 (GPU)
d_arr_a = cuda.to_device(arr_a)
d_arr_b = cuda.to_device(arr_b)
d_result_gpu = cuda.device_array(size_cuda, dtype=np.float32) # 在 GPU 上分配輸出空間

# 設定 CUDA 核心的執行配置 (Grid and Block Dimensions)
threads_per_block = 256
blocks_per_grid = (size_cuda + (threads_per_block - 1)) // threads_per_block

start_time_gpu = time.time()
# 啟動 CUDA 核心
vector_add_gpu[blocks_per_grid, threads_per_block](d_arr_a, d_arr_b, d_result_gpu)
cuda.synchronize() # 等待 GPU 任務完成
end_time_gpu = time.time()

# 將結果從裝置 (GPU) 複製回主機 (CPU)
result_gpu = d_result_gpu.copy_to_host()

print(f"Numba CUDA 執行時間: {end_time_gpu - start_time_gpu:.4f} 秒")

# 驗證結果
print(f"結果是否近似一致: {np.allclose(result_cpu, result_gpu)}")

# 清理 GPU 記憶體 (可選，通常在 Python 中會自動處理)
del d_arr_a, d_arr_b, d_result_gpu


Numba 已找到 CUDA GPU。

正在執行純 NumPy (CPU) 向量加法...
純 NumPy 執行時間: 0.0155 秒

正在執行 Numba CUDA (GPU) 向量加法...
Numba CUDA 執行時間: 1.6371 秒
結果是否近似一致: True


#### 程式說明：

-   **`cuda.is_available()`**: 檢查 Colab 環境中是否有可用的 NVIDIA GPU。
-   **`@cuda.jit`**: 這是 Numba CUDA 的關鍵裝飾器，它將 Python 函數標記為一個可以在 GPU 上運行的 CUDA 核心。
-   **`idx = cuda.grid(1)`**: 在 CUDA 核心中，每個執行緒都會運行這個函數。`cuda.grid(1)` 函數會計算當前執行緒在 1 維網格中的唯一全局索引。我們使用這個索引來訪問數組的特定元素。
-   **`cuda.to_device(arr)`**: 將 NumPy 數組 `arr` 從 CPU 記憶體複製到 GPU 記憶體。GPU 只能直接存取其自身的記憶體。
-   **`cuda.device_array(size, dtype)`**: 在 GPU 記憶體上分配一個指定大小和資料型別的空白數組，用於存放計算結果。
-   **`threads_per_block` 和 `blocks_per_grid`**: 這是 CUDA 的執行配置。GPU 核心是透過網格（grid）和區塊（block）組織的。每個區塊包含固定數量的執行緒（`threads_per_block`），而網格包含固定數量的區塊（`blocks_per_grid`）。這些參數決定了有多少個執行緒將同時運行您的 CUDA 核心。
    -   `blocks_per_grid = (size_cuda + (threads_per_block - 1)) // threads_per_block` 是一種常見的計算方式，用於確保所有元素都能被處理，即使資料大小不是 `threads_per_block` 的倍數。
-   **`vector_add_gpu[blocks_per_grid, threads_per_block](d_arr_a, d_arr_b, d_result_gpu)`**: 這是啟動 CUDA 核心的方式。方括號中的 `[blocks_per_grid, threads_per_block]` 定義了核心的啟動配置，後面的圓括號則是傳遞給 CUDA 核心的參數（必須是 GPU 上的資料）。
-   **`cuda.synchronize()`**: CUDA 核心是異步執行的，這意味著 CPU 會在核心啟動後立即繼續執行。`cuda.synchronize()` 會阻塞 CPU 的執行，直到所有 GPU 上的任務完成，這樣我們才能確保結果已經準備好被複製回 CPU。
-   **`d_result_gpu.copy_to_host()`**: 將計算完成的結果從 GPU 記憶體複製回 CPU 記憶體，以便在 Python 中進行後續處理或驗證。

您會發現，即使包含了資料複製的時間，GPU 版本對於大型數組的運算速度通常會比 CPU NumPy 版本快很多，這突顯了 GPU 在平行處理上的優勢。

## 對話紀錄總結

本次對話主要圍繞著 **Numba 教學**，逐步介紹了 Numba 的各種功能。

1.  **安裝 Numba**: 透過 `pip install numba` 進行安裝。
2.  **基本使用 (`@jit`)**: 示範了如何使用 `@jit` 裝飾器加速 Python 函數，並比較了純 Python 與 Numba 加速後的效能差異。
    *   **結果**: Numba 加速後性能提升約 **120 倍**。
3.  **元素級運算 (`@vectorize`)**: 介紹了如何使用 `@vectorize` 裝飾器將標量函數轉換為 NumPy universal function (ufunc)，以加速元素級運算。
    *   **結果**: Numba `vectorize` 加速後性能提升約 **162 倍**。
4.  **平行運算 (`prange`)**: 講解了如何使用 `prange` 配合 `@jit(parallel=True)` 啟用 CPU 平行運算。
    *   **結果**: `prange` 版本相較於非平行 Numba 版本，性能提升超過 **20 倍**。
5.  **GPU 加速 (`cuda`)**: 示範了 Numba CUDA 的基本用法，包括定義 CUDA 核心、資料在 CPU 和 GPU 間的傳輸、設定執行配置等。
    *   **目前狀態**: 在執行 CUDA 範例時遇到 `CudaSupportError`，因為 Colab 執行階段未啟用 GPU。已提供使用者啟用 GPU 的指示。

#### 程式說明：

-   `from numba import jit`: 從 `numba` 庫中導入 `jit` 裝飾器。
-   `@jit(nopython=True)`: 這是 Numba 的核心。將這個裝飾器應用於 `sum_square_numba` 函數上方，告訴 Numba 在運行時將此函數編譯為機器碼。
    -   `nopython=True` 是一個非常重要的參數。它強制 Numba 將整個函數編譯成不依賴於 Python C API 的機器碼。如果 Numba 無法將函數完全編譯為機器碼（例如，函數內部使用了 Numba 無法理解的 Python 特性），它會拋出錯誤。這通常能帶來最佳的效能提升。
-   `sum_square_numba(a)`: 此函數的邏輯與 `sum_square_pure_python` 相同，但由於 `jit` 裝飾器，其執行方式將完全不同。
-   **首次調用與再次調用**: 請注意，第一次調用 `sum_square_numba(arr)` 時，Numba 會執行 JIT 編譯過程，這會花費一些時間。因此，首次調用的時間會包含編譯時間。然而，一旦函數被編譯，後續的調用將直接使用已編譯的機器碼，其執行速度會快得多，這就是 Numba 帶來效能提升的關鍵。

### 效能比較總結

運行上述程式碼後，您會觀察到 Numba 優化後的函數在第二次運行時，執行時間會顯著縮短，通常比純 Python 函數快數十甚至數百倍。這正是 Numba 的強大之處，它讓您能夠用 Python 的便利性編寫程式，卻能獲得接近原生代碼的執行效能，特別是在數值密集型迴圈中。